# SharePoint Tenant Governance Report

Interactive tenant health dashboard covering site inventory, sharing exposure,
stale content, external users, hub topology, and archived sites.

> **Requires:** tenant admin credentials with `SharePoint Administrator` role.
> **Docs:** [SPO Tenant Admin REST API](https://learn.microsoft.com/en-us/sharepoint/dev/apis/sharepoint-rest-api-index)

---


## 0 · Setup

In [1]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone

import pandas as pd

#from IPython.display import display
from office365.sharepoint.client_context import ClientContext
from office365.sharepoint.tenant.administration.sharing_capabilities import SharingCapabilities
from office365.sharepoint.tenant.administration.tenant import Tenant
from office365.sharepoint.tenant.management.office365_tenant import Office365Tenant
from tests import test_admin_site_url, test_client_id, test_client_secret

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

# ── Auth ───────────────────────────────────────────────────────────────────────
ctx = ClientContext(test_admin_site_url).with_client_credentials(test_client_id, test_client_secret)
tenant = Tenant(ctx)
o365 = Office365Tenant(ctx)

SHARING_LABELS = {
    SharingCapabilities.Disabled: "Disabled",
    SharingCapabilities.ExternalUserSharingOnly: "Authenticated external only",
    SharingCapabilities.ExternalUserAndGuestSharing: "Anyone (links + guests)",
    SharingCapabilities.ExistingExternalUserSharingOnly: "Existing guests only",
}

STALE_DAYS = 180  # sites with no content change beyond this are flagged

print(f"Connected to: {test_admin_site_url}")
print(f"Report generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

Connected to: https://mediadev8-admin.sharepoint.com
Report generated: 2026-05-23 17:18 UTC


---
## 1 · Site Inventory

Full list of SharePoint site collections with owner, sharing policy, lock state,
hub membership, and last content change. This is the foundation for every other
section of the report.


In [2]:
sites = tenant.get_site_properties_from_sharepoint_by_filters(include_detail=True).execute_query()

now = datetime.now(timezone.utc)

rows = []
for s in sites:
    last_modified = s.last_content_modified_date
    # last_content_modified_date returns datetime.min when not set
    if last_modified and last_modified.year > 2000:
        lm_aware = last_modified.replace(tzinfo=timezone.utc) if last_modified.tzinfo is None else last_modified
        days_since = (now - lm_aware).days
    else:
        days_since = None

    rows.append(
        {
            "Title": s.title or "(no title)",
            "URL": s.url,
            "Owner": s.owner_login_name,
            "Sharing": SHARING_LABELS.get(s.sharing_capability, str(s.sharing_capability)),
            "Lock": s.lock_state or "Unlock",
            "Hub": "✓" if s.is_hub_site else "",
            "Days since change": days_since,
            "Archived": "✓" if s.archived_by else "",
        }
    )

df_sites = pd.DataFrame(rows)
print(f"Total sites: {len(df_sites)}")
display(df_sites)

Unhandled exception in listener <bound method AuthenticationContext.authenticate_request of <office365.runtime.auth.authentication_context.AuthenticationContext object at 0x7d76e15d6710>>
Traceback (most recent call last):
  File "/home/vgrem/dev/office365-rest-python-client/office365/runtime/auth/providers/acs_token_provider.py", line 53, in get_app_only_access_token
    return self._get_app_only_access_token(url_info.hostname, realm)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vgrem/dev/office365-rest-python-client/office365/runtime/auth/providers/acs_token_provider.py", line 81, in _get_app_only_access_token
    response.raise_for_status()
  File "/home/vgrem/dev/office365-rest-python-client/.venv/lib/python3.11/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://login.microsoftonline.com/af6a80a4-8b4b-

ClientRequestException: ('-2147024891, System.UnauthorizedAccessException', 'Access is denied. (Exception from HRESULT: 0x80070005 (E_ACCESSDENIED))', '403 Client Error: FORBIDDEN for url: https://mediadev8-admin.sharepoint.com/_api/Microsoft.Online.SharePoint.TenantAdministration.Tenant/getSitePropertiesFromSharePointByFilters')

---
## 2 · Sharing Exposure

Sites grouped by sharing capability. `Anyone (links + guests)` is the highest
risk category — anyone with the link can access content without authentication.


In [ ]:
sharing_summary = df_sites.groupby("Sharing").size().reset_index(name="Count").sort_values("Count", ascending=False)
display(sharing_summary)

# High-risk sites: Anyone link sharing enabled
high_risk = df_sites[df_sites["Sharing"] == "Anyone (links + guests)"]
print(f"\n⚠️  High-risk sites (anyone link sharing): {len(high_risk)}")
if len(high_risk):
    display(high_risk[["Title", "URL", "Owner", "Lock"]])

---
## 3 · Stale Sites (>{STALE_DAYS} days since last content change)

Candidates for archiving, lifecycle review, or deletion.
`Days since change` is based on `LastContentModifiedDate` from the admin API.
Sites with no recorded modification date are listed separately.


In [ ]:
df_with_date = df_sites[df_sites["Days since change"].notna()].copy()
df_no_date = df_sites[df_sites["Days since change"].isna()].copy()

stale = df_with_date[df_with_date["Days since change"] > STALE_DAYS].sort_values("Days since change", ascending=False)

print(f"Stale sites (>{STALE_DAYS} days): {len(stale)}")
print(f"Sites with no modification date:  {len(df_no_date)}")

display(stale[["Title", "URL", "Owner", "Days since change", "Sharing", "Archived"]])

---
## 4 · Locked Sites

Sites in `ReadOnly` or `NoAccess` lock state. These are blocked from normal
user operations — usually due to storage quota breach, compliance hold, or
manual administrative action.


In [ ]:
locked = df_sites[df_sites["Lock"].isin(["ReadOnly", "NoAccess"])].copy()
print(f"Locked sites: {len(locked)}")
if len(locked):
    display(locked[["Title", "URL", "Owner", "Lock", "Sharing"]])
else:
    print("✅ No locked sites found.")

---
## 5 · Archived Sites

Sites that have been placed in archive state. Archived sites are read-only and
count against storage but not against active site limits.


In [ ]:
archived = df_sites[df_sites["Archived"] == "✓"].copy()
print(f"Archived sites: {len(archived)}")
if len(archived):
    display(archived[["Title", "URL", "Owner", "Days since change"]])
else:
    print("No archived sites.")

---
## 6 · Hub Site Topology

Hub sites and their registered details. Hub sites are the backbone of
SharePoint information architecture — orphaned or mis-configured hubs
create navigation and search fragmentation.


In [ ]:
hub_sites = tenant.get_hub_sites().execute_query()

hub_rows = []
for h in hub_sites:
    hub_rows.append(
        {
            "Title": h.title,
            "Site URL": h.site_url,
            "Hub ID": str(h.id),
        }
    )

df_hubs = pd.DataFrame(hub_rows) if hub_rows else pd.DataFrame(columns=["Title", "Site URL", "Hub ID"])
print(f"Hub sites: {len(df_hubs)}")
display(df_hubs)

# Cross-reference: which inventory sites are hub members?
hub_members = df_sites[df_sites["Hub"] == "✓"][["Title", "URL", "Owner"]]
print(f"\nSites flagged as hub sites in inventory: {len(hub_members)}")
display(hub_members)

---
## 7 · External Users

Guest and external accounts with access to your tenant. Returned in pages of
up to 50; this cell iterates all pages. High external user count combined with
`Anyone` sharing is the most common over-exposure pattern.


In [ ]:
all_external = []
position = 0
page_size = 50

while True:
    result = o365.get_external_users(position=position, page_size=page_size).execute_query()
    users = result.external_user_collection
    if not users:
        break
    for u in users:
        all_external.append(
            {
                "Display name": u.display_name,
                "Email": u.invited_as or u.login_name,
                "Accepted": u.accepted_as,
                "When created": u.when_created,
                "Invited by": u.invited_by,
            }
        )
    if len(users) < page_size:
        break
    position += page_size

df_external = pd.DataFrame(all_external)
print(f"Total external users: {len(df_external)}")
if len(df_external):
    display(df_external.sort_values("Email"))
else:
    print("✅ No external users found.")

---
## 8 · Governance Summary

Single-glance health scorecard. Use this as the executive summary or as a
baseline to track improvement over time.


In [ ]:
total = len(df_sites)
high_risk_n = len(df_sites[df_sites["Sharing"] == "Anyone (links + guests)"])
stale_n = len(stale)
locked_n = len(locked)
archived_n = len(archived)
hub_n = len(df_hubs)
external_n = len(df_external)

summary = pd.DataFrame(
    [
        {"Metric": "Total site collections", "Value": total},
        {"Metric": "High-risk sharing (anyone links)", "Value": high_risk_n},
        {"Metric": f"Stale sites (>{STALE_DAYS} days)", "Value": stale_n},
        {"Metric": "Locked sites", "Value": locked_n},
        {"Metric": "Archived sites", "Value": archived_n},
        {"Metric": "Hub sites", "Value": hub_n},
        {"Metric": "External / guest users", "Value": external_n},
    ]
)

display(summary.style.hide(axis="index"))

risk_pct = round(high_risk_n / total * 100, 1) if total else 0
stale_pct = round(stale_n / total * 100, 1) if total else 0
print(f"\nRisk exposure:  {risk_pct}% of sites allow anyone-link sharing")
print(f"Stale content:  {stale_pct}% of sites have not changed in >{STALE_DAYS} days")